<a href="https://colab.research.google.com/github/clobos/Python_Gemini_Colab/blob/main/AM_classificacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Preparação do Ambiente e Geração de Dados Sintéticos

Primeiro, vamos importar as bibliotecas necessárias e criar um conjunto de dados sintético para um problema de classificação binária usando `sklearn.datasets.make_classification`.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc, confusion_matrix,
                             RocCurveDisplay)

# Gerar um conjunto de dados sintético para classificação binária
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5,
                           n_classes=2, random_state=42)

df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
df['target'] = y

print(f"Shape do dataset: {df.shape}")
display(df.head())

Shape do dataset: (1000, 21)


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,target
0,1.470848,-0.360450,-0.591602,-0.728228,0.941690,1.065964,0.017832,-0.596184,1.840712,-1.497093,...,-0.603968,2.899256,0.037567,-1.249523,0.257963,0.416628,1.408208,-1.838041,-0.833142,1
1,4.513369,-2.227103,-1.140747,2.018263,-2.238358,-0.497370,0.714550,0.938883,-2.395169,0.159837,...,1.461499,3.954171,0.309054,0.538184,-7.157865,-4.532216,-0.081800,-9.325362,0.574386,1
2,-2.355643,2.218601,-1.603269,0.873394,0.401483,0.717264,-0.859399,-1.042190,-2.175965,0.980231,...,0.544434,-2.466258,-0.470256,0.073018,-2.203531,-2.299263,-1.742761,-0.271579,-0.359285,0
3,-1.596198,-0.857427,1.772434,-0.639361,1.419409,-0.438525,0.281949,2.345145,1.006230,0.389135,...,-1.025051,-2.422975,1.579807,-0.300713,4.267120,2.893775,1.236697,6.034785,-0.045711,0
4,2.840049,-2.489600,-0.844902,-1.594362,-4.688517,0.459637,0.913607,-1.143505,1.263937,-2.040928,...,4.176424,1.341742,0.133565,1.743819,1.531188,2.269808,0.053489,-3.151109,1.603702,0


### 2. Configuração da Validação Cruzada e Modelos

Vamos configurar a validação cruzada k-fold estratificada (k=5) e inicializar os modelos que serão comparados: Regressão Logística, Árvore de Decisão e Floresta Aleatória.

In [2]:
# Configurar validação cruzada k-fold estratificada
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Definir os modelos
models = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# Dicionários para armazenar métricas e resultados por modelo
metrics_results = {name: {'accuracy': [], 'precision': [], 'recall': [], 'specificity': [], 'f1_score': [], 'fpr': [], 'tpr': [], 'auc': []} for name in models.keys()}
confusion_matrices = {name: np.zeros((2, 2)) for name in models.keys()}

### 3. Treinamento e Avaliação dos Modelos com Validação Cruzada

Agora, vamos iterar sobre cada fold da validação cruzada, treinar cada modelo, fazer previsões e calcular as métricas de avaliação. As métricas incluem acurácia, precisão, recall (sensibilidade), especificidade, F1-score, e os dados para a curva ROC (FPR, TPR e AUC).

In [3]:
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    print(f"\n--- Fold {fold+1}/{kf.n_splits} ---")

    for name, model in models.items():
        print(f"Treinando e avaliando: {name}")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] # Probabilidades para a curva ROC

        # Métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Matriz de Confusão para Especificidade
        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp)

        # Curva ROC e AUC
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)

        # Armazenar resultados
        metrics_results[name]['accuracy'].append(acc)
        metrics_results[name]['precision'].append(prec)
        metrics_results[name]['recall'].append(rec)
        metrics_results[name]['specificity'].append(specificity)
        metrics_results[name]['f1_score'].append(f1)
        metrics_results[name]['fpr'].append(fpr)
        metrics_results[name]['tpr'].append(tpr)
        metrics_results[name]['auc'].append(roc_auc)

        # Somar matrizes de confusão para média (arredondar para o inteiro mais próximo)
        confusion_matrices[name] += cm

# Calcular a média das matrizes de confusão ao longo dos folds
for name in confusion_matrices:
    confusion_matrices[name] = np.round(confusion_matrices[name] / kf.n_splits)

# Exibir métricas médias por modelo
print("\n--- Métricas Médias por Modelo ---")
for name, results in metrics_results.items():
    print(f"\nModelo: {name}")
    for metric, values in results.items():
        if metric not in ['fpr', 'tpr', 'auc']:
            print(f"  {metric.replace('_', ' ').capitalize()}: {np.mean(values):.4f}")


--- Fold 1/5 ---
Treinando e avaliando: Logistic Regression
Treinando e avaliando: Decision Tree
Treinando e avaliando: Random Forest

--- Fold 2/5 ---
Treinando e avaliando: Logistic Regression
Treinando e avaliando: Decision Tree
Treinando e avaliando: Random Forest

--- Fold 3/5 ---
Treinando e avaliando: Logistic Regression
Treinando e avaliando: Decision Tree
Treinando e avaliando: Random Forest

--- Fold 4/5 ---
Treinando e avaliando: Logistic Regression
Treinando e avaliando: Decision Tree
Treinando e avaliando: Random Forest

--- Fold 5/5 ---
Treinando e avaliando: Logistic Regression
Treinando e avaliando: Decision Tree
Treinando e avaliando: Random Forest

--- Métricas Médias por Modelo ---

Modelo: Logistic Regression
  Accuracy: 0.8420
  Precision: 0.8392
  Recall: 0.8509
  Specificity: 0.8332
  F1 score: 0.8445

Modelo: Decision Tree
  Accuracy: 0.8320
  Precision: 0.8185
  Recall: 0.8568
  Specificity: 0.8068
  F1 score: 0.8369

Modelo: Random Forest
  Accuracy: 0.9330
 

### 4. Visualização das Matrizes de Confusão Médias

Vamos visualizar as matrizes de confusão médias para cada modelo usando Plotly para entender melhor como cada um classifica corretamente as classes positivas e negativas.

In [4]:
for name, cm in confusion_matrices.items():
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] # Normalizar para porcentagem

    fig = px.imshow(cm_norm, text_auto=True, color_continuous_scale='Viridis',
                    labels=dict(x="Previsão", y="Real", color="Intensidade"),
                    x=['Negativo (0)', 'Positivo (1)'],
                    y=['Negativo (0)', 'Positivo (1)'],
                    title=f'Matriz de Confusão Média - {name} (Normalizada)')

    fig.update_layout(xaxis_title='Previsão', yaxis_title='Real')
    fig.show()

### 5. Visualização das Curvas ROC Médias

Em seguida, vamos plotar as curvas ROC médias para cada modelo, juntamente com o valor médio da AUC (Area Under the Curve), para avaliar sua capacidade de discriminação entre as classes.

In [5]:
fig_roc = go.Figure()
fig_roc.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)

for name, results in metrics_results.items():
    # Calcular FPR e TPR médios
    all_fpr = np.unique(np.concatenate([results['fpr'][i] for i in range(len(results['fpr']))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(results['tpr'])):
        mean_tpr += np.interp(all_fpr, results['fpr'][i], results['tpr'][i])
    mean_tpr /= len(results['tpr'])
    mean_auc = np.mean(results['auc'])

    fig_roc.add_trace(go.Scatter(x=all_fpr, y=mean_tpr, name=f'{name} (AUC = {mean_auc:.2f})', mode='lines'))

fig_roc.update_layout(
    title='Curva ROC Média para Cada Modelo',
    xaxis_title='Taxa de Falsos Positivos (FPR)',
    yaxis_title='Taxa de Verdadeiros Positivos (TPR)',
    yaxis=dict(scaleanchor="x", scaleratio=1)
)
fig_roc.show()

### 6. Qual o Melhor Modelo em Termos de Capacidade Preditiva?

Com base nas métricas e nas visualizações, podemos agora determinar o melhor modelo.

In [6]:
best_model_name = max(metrics_results, key=lambda name: np.mean(metrics_results[name]['accuracy']))

print(f"Com base na acurácia média, o melhor modelo é: {best_model_name}")
print(f"Métricas médias para o {best_model_name}:")
for metric, values in metrics_results[best_model_name].items():
    if metric not in ['fpr', 'tpr', 'auc']:
        print(f"  {metric.replace('_', ' ').capitalize()}: {np.mean(values):.4f}")
    elif metric == 'auc':
        print(f"  AUC: {np.mean(values):.4f}")

print("\nO modelo Random Forest se destacou com a maior acurácia, precisão, recall, especificidade, F1-score e AUC média, indicando uma capacidade preditiva superior para este conjunto de dados.\nEle demonstrou a melhor capacidade de distinguir entre as classes positiva e negativa, com um bom equilíbrio entre a identificação correta de positivos e a minimização de falsos positivos e negativos.")

Com base na acurácia média, o melhor modelo é: Random Forest
Métricas médias para o Random Forest:
  Accuracy: 0.9330
  Precision: 0.9314
  Recall: 0.9364
  Specificity: 0.9296
  F1 score: 0.9333
  AUC: 0.9794

O modelo Random Forest se destacou com a maior acurácia, precisão, recall, especificidade, F1-score e AUC média, indicando uma capacidade preditiva superior para este conjunto de dados.
Ele demonstrou a melhor capacidade de distinguir entre as classes positiva e negativa, com um bom equilíbrio entre a identificação correta de positivos e a minimização de falsos positivos e negativos.
